In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, ParameterGrid
from tqdm import tqdm

# Load Cleaned Dataset from Data.ipynb-File
data = pd.read_csv("data/Clean_NY.csv")  # Read data in variable
X = data.drop("PRICE", axis=1)  # Extracts Independent Training Data within Training-Dataframe
y = data["PRICE"].astype('float32')  # Converts Dependent Variable in float32-Format

# Split in Training-, Validation- and Testdata
# train-data for GRID training
# val-data for GRID validation 
# Train-date (train- & val-data) for final model training 
# test-data for final evaluation
# Split Train/Val/Test = 0.7/0,15/0,15
X_Train, X_test, y_Train, y_test = train_test_split(X, y, test_size=0.15, random_state=40)
X_train, X_val, y_train, y_val = train_test_split(X_Train, y_Train, test_size=0.15/0.85, random_state=42)

# The following code uses GRID-Search for Parameter optimization and trying all possible combinations below

# Grid-Search Parameter
param_grid = {
    'criterion': ['squared_error'],
    'splitter': ['best'],
    'max_depth': [None, 10, 20, 50, 100],
    'min_samples_split': [2, 3, 4, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'min_impurity_decrease': [0.0, 0.01, 0.1]
}

# Create List of all Parameter-Combinations
param_combinations = list(ParameterGrid(param_grid))

# Initialise Progress Bar
progress_bar = tqdm(total=len(param_combinations))

# Initialise preset variables for best model scores (for specific parameters combination) and extra variable for this parameter combination
best_score = -float('inf')
best_params = None

# Implement GRID-Search manually
# The problem with the classic GridSearchCV-Method is you have no idea how the calculation will take. You can wait forever without 
# knowing before if you put to many parameter combinations into the method. 
# With the progress bar you get some clue how many parameter combinations are manageable for the specific machine 
for params in param_combinations:
    progress_bar.update(1)  # # Update progress bar
    model = DecisionTreeRegressor(random_state=42, **params)
    model.fit(X_train, y_train)
    score = r2_score(y_val, model.predict(X_val))
    
    if score > best_score:
        best_score = score
        best_params = params

progress_bar.close()

# Print results
print("Best Parameters: \n", best_params)
print(f"Best R²: {best_score:.4f}")

100%|█████████████████████████████████████████| 360/360 [00:06<00:00, 55.59it/s]

Best Parameters: 
 {'criterion': 'squared_error', 'max_depth': 100, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 20, 'splitter': 'best'}
Best R²: 0.5319


In [2]:
# Print best parameters again
print("Best Parameters:", best_params)

# Create final model
dt_model = DecisionTreeRegressor(random_state=42, **best_params)


# Train final Model with full training data including train-data and val-data
dt_model.fit(X_Train, y_Train)

# Predictions with final model with best parameters on test-set
y_pred = dt_model.predict(X_test)
# Predictions with final model with best parameters on training-set
y_pred_train = dt_model.predict(X_Train)

# Calculate metrics to evaluate the model's performance
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2_train = r2_score(y_Train, y_pred_train)
r2 = r2_score(y_test, y_pred)

# Print metrics
print(f"Decision Tree R² Train: {r2_train:.4f}")
print(f"Decision Tree R² Test: {r2:.4f}")
print(f"Decision Tree MSE: {mse:.4f}")
print(f"Decision Tree MAE: {mae:.4f}")

Best Parameters: {'criterion': 'squared_error', 'max_depth': 100, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 20, 'splitter': 'best'}
Decision Tree R² Train: 0.8011
Decision Tree R² Test: 0.6050
Decision Tree MSE: 2492105247172.7095
Decision Tree MAE: 697737.4346


In [3]:
# Save and export model via pickle framework
import pickle

with open('dt_model.pkl', 'wb') as file:
    pickle.dump(dt_model, file)
